# OpenVLA — Step 3: Evaluation

This notebook evaluates the fine-tuned OpenVLA model using L1/L2 action prediction error.

**Prerequisites:**
- Run `01_data_preparation.ipynb` (local dataset at `./bridge_hf_synthetic`)
- Run `02_training_job.ipynb` and download the model artifacts
- GPU instance with 48GB+ VRAM

**What this does:**
1. Evaluates the fine-tuned checkpoint on validation data
2. Evaluates the base model for comparison
3. Compares results (L1/L2 error)

## 1. Environment Setup

In [2]:
# !pip install -q torch transformers==4.40.1 tokenizers==0.19.1 peft==0.12.0 accelerate pillow numpy tqdm
# !pip install -q "timm>=0.9.10,<1.0.0"
# !pip install -q "huggingface_hub==0.25.2" "datasets==4.5.0"
!pip install -r scripts/requirements.txt 
#!pip install --no-build-isolation "flash-attn==2.5.5" 2>/dev/null || echo 'flash-attn install skipped'

  Using cached torch-2.7.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.22.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.6.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.6.80-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.5.1.17-py3-none-manylinux_2_28_x86_64.whl.metadata (1.6 kB)
  Using cached nvid

In [3]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Enter your Hugging Face token: ')
login(token=hf_token)

Enter your Hugging Face token:  ········


## 2. Configuration

In [4]:
import os, sys, json, glob, boto3
import numpy as np
import torch
from pathlib import Path
from datasets import load_from_disk
from transformers import AutoProcessor, AutoModelForVision2Seq
from tqdm import tqdm

# Add scripts dir to path for utils
sys.path.insert(0, str(Path('./scripts').absolute()))
from utils.openvla_utils import ActionTokenizer, PurePromptBuilder, VicunaV15ChatPromptBuilder

# Auto-detect the latest completed training job
sm_client = boto3.client('sagemaker')
job_resp = sm_client.list_training_jobs(
    NameContains='openvla-lora-finetune-bridge',
    SortBy='CreationTime', SortOrder='Descending', MaxResults=10)
completed_jobs = [j for j in job_resp['TrainingJobSummaries'] if j['TrainingJobStatus'] == 'Completed']
if completed_jobs:
    TRAINING_JOB_NAME = completed_jobs[0]['TrainingJobName']
    print(f'Auto-detected training job: {TRAINING_JOB_NAME}')
else:
    TRAINING_JOB_NAME = 'UNKNOWN'
    print('WARNING: No completed training jobs found. Set TRAINING_JOB_NAME manually.')

# Model paths — use absolute paths for local models
FINETUNED_MODEL_PATH = str(Path(f'./model_artifacts/{TRAINING_JOB_NAME}/extracted/openvla_lora_finetune').absolute())
BASE_MODEL_PATH = 'openvla/openvla-7b'
DATASET_PATH = './bridge_hf_synthetic'
NUM_SAMPLES = 100

# Check if the path exists, try alternate names
if not os.path.isdir(FINETUNED_MODEL_PATH):
    alt = str(Path(f'./model_artifacts/{TRAINING_JOB_NAME}/extracted').absolute())
    candidates = glob.glob(alt + '/*') if os.path.isdir(alt) else []
    if candidates:
        FINETUNED_MODEL_PATH = candidates[0]
        print(f'Using model dir: {FINETUNED_MODEL_PATH}')
    else:
        print(f'WARNING: Model not found at expected path. Contents of model_artifacts:')
        !find ./model_artifacts -maxdepth 4 -type d 2>/dev/null | head -20

print(f'Fine-tuned model: {FINETUNED_MODEL_PATH}')
print(f'Base model: {BASE_MODEL_PATH}')
print(f'Dataset: {DATASET_PATH}')

2026-04-20 22:33:42.710101: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776724422.722817    5911 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776724422.726649    5911 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-20 22:33:42.739925: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Auto-detected training job: openvla-lora-finetune-bridge-20260416163449
Fine-tuned model: /home/sagemaker-user/OpenVLA_SMTJ/model_artifacts/openvla-lora-finetune-bridge-20260416163449/extracted/openvla_lora_finetune
Base model: openvla/openvla-7b
Dataset: ./bridge_hf_synthetic


## 3. Evaluation Function

In [5]:
def evaluate_model(model_path, dataset_path, num_samples=100, base_model_path='openvla/openvla-7b'):
    """Evaluate an OpenVLA model and return L1/L2 error metrics."""
    print(f'Loading model from {model_path}')
    # Always load processor from base model (tokenizer is unchanged by LoRA fine-tuning)
    processor = AutoProcessor.from_pretrained(base_model_path, trust_remote_code=True)
    model = AutoModelForVision2Seq.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        attn_implementation='eager', trust_remote_code=True,
    ).to('cuda')
    model.eval()
    action_tokenizer = ActionTokenizer(processor.tokenizer)

    dataset = load_from_disk(dataset_path)
    val_dataset = dataset['validation'] if 'validation' in dataset else dataset['train']
    num_samples = min(num_samples, len(val_dataset))
    indices = np.random.choice(len(val_dataset), num_samples, replace=False)

    l1_errors, l2_errors = [], []
    with torch.no_grad():
        for idx in tqdm(indices, desc='Evaluating'):
            sample = val_dataset[int(idx)]
            image = sample['observation/image'][0]
            action_gt = np.array(sample['actions'][0])
            instruction = sample['language_instruction']

            prompt_text = PurePromptBuilder('openvla').build_prompt(instruction, '')
            inputs = processor(text=prompt_text, images=image, return_tensors='pt').to('cuda', dtype=torch.bfloat16)
            outputs = model(**inputs)

            action_logits = outputs.logits[0]
            action_token_ids = torch.argmax(action_logits[-7:], dim=-1).cpu().numpy()
            action_pred = action_tokenizer.decode_token_ids_to_actions(action_token_ids)

            l1_errors.append(np.mean(np.abs(action_pred - action_gt)))
            l2_errors.append(np.sqrt(np.mean((action_pred - action_gt) ** 2)))

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

    return {
        'mean_l1': float(np.mean(l1_errors)),
        'median_l1': float(np.median(l1_errors)),
        'std_l1': float(np.std(l1_errors)),
        'mean_l2': float(np.mean(l2_errors)),
        'median_l2': float(np.median(l2_errors)),
        'std_l2': float(np.std(l2_errors)),
    }

## 4. Evaluate Fine-Tuned Model

In [6]:
# !pip uninstall -y peft && pip install peft==0.14.0 --no-cache-dir --force-reinstall
# !grep "MODEL_TYPE_TO_PEFT_MODEL_MAPPING" /opt/conda/lib/python3.12/site-packages/peft/mapping.py
# !find /opt/conda -path "*/peft/mapping.py" -exec grep -l "MODEL_TYPE_TO_PEFT_MODEL_MAPPING" {} \;
# !find /opt/conda -name "peft" -type d



In [7]:
np.random.seed(42)
finetuned_results = evaluate_model(FINETUNED_MODEL_PATH, DATASET_PATH, NUM_SAMPLES)
print(f"\nFine-tuned — L1 Mean: {finetuned_results['mean_l1']:.4f}, L2 Mean: {finetuned_results['mean_l2']:.4f}")

Loading model from /home/sagemaker-user/OpenVLA_SMTJ/model_artifacts/openvla-lora-finetune-bridge-20260416163449/extracted/openvla_lora_finetune


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/opt/conda/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Expected `transformers==4.40.1` and `tokenizers==0.19.1` but got `transformers==4.57.6` and `tokenizers==0.22.2`; there might be inference-time regressions due to dependency changes. If in doubt, pleaseuse the above versions.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 60/60 [00:05<00:00, 11.73it/s]



Fine-tuned — L1 Mean: 0.1769, L2 Mean: 0.3277


## 5. Evaluate Base Model (for comparison)

In [8]:
np.random.seed(42)
base_results = evaluate_model(BASE_MODEL_PATH, DATASET_PATH, NUM_SAMPLES)
print(f"\nBase — L1 Mean: {base_results['mean_l1']:.4f}, L2 Mean: {base_results['mean_l2']:.4f}")

Loading model from openvla/openvla-7b


Expected `transformers==4.40.1` and `tokenizers==0.19.1` but got `transformers==4.57.6` and `tokenizers==0.22.2`; there might be inference-time regressions due to dependency changes. If in doubt, pleaseuse the above versions.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 60/60 [00:04<00:00, 14.60it/s]



Base — L1 Mean: 0.2925, L2 Mean: 0.3967


## 6. Results Comparison

In [9]:
print('=' * 60)
print('EVALUATION RESULTS')
print('=' * 60)
print(f"{'Metric':<20} {'Base Model':<15} {'Fine-Tuned':<15} {'Improvement':<15}")
print('-' * 60)
for metric in ['mean_l1', 'median_l1', 'std_l1', 'mean_l2', 'median_l2', 'std_l2']:
    base_val = base_results[metric]
    ft_val = finetuned_results[metric]
    improvement = (base_val - ft_val) / base_val * 100 if base_val > 0 else 0
    print(f'{metric:<20} {base_val:<15.4f} {ft_val:<15.4f} {improvement:>+.1f}%')

# Save results
results = {'base': base_results, 'finetuned': finetuned_results, 'num_samples': NUM_SAMPLES}
with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to evaluation_results.json')

EVALUATION RESULTS
Metric               Base Model      Fine-Tuned      Improvement    
------------------------------------------------------------
mean_l1              0.2925          0.1769          +39.5%
median_l1            0.2906          0.1571          +45.9%
std_l1               0.0739          0.0610          +17.4%
mean_l2              0.3967          0.3277          +17.4%
median_l2            0.3910          0.2982          +23.7%
std_l2               0.0775          0.0675          +12.8%

Results saved to evaluation_results.json
